# IDP Lesson 01: AI Parsing

In [ ]:
CREATE OR REPLACE TEMP VIEW raw_unstructured_doc AS
SELECT path, content
FROM read_files('/Volumes/idp/default/ai_parsing_lesson')

In [ ]:
SELECT 
path, length(content) 
FROM raw_unstructured_doc

In [ ]:
CREATE OR REPLACE TEMP VIEW parsed_structured_doc AS
SELECT 
path, 
ai_parse_document(content) AS parsed_content
FROM raw_unstructured_doc

In [ ]:
SELECT path, parsed_content
FROM parsed_structured_doc

In [ ]:
CREATE OR REPLACE TEMP VIEW structured_tables AS
SELECT path, 
try_cast(e:content AS STRING) as table_html
FROM parsed_structured_doc
LATERAL VIEW explode(try_cast(parsed_content:document:elements AS ARRAY<VARIANT>)) t AS e
WHERE try_cast(e:type AS STRING) = 'table'

In [ ]:
SELECT *
FROM structured_tables

## 2.0 To Extract the Specific Information Required

In [ ]:
SELECT *,
ai_extract(table_html, array('CPT Code')).`CPT Code` AS CPT_Code,
ai_extract(table_html, array('ICD Code')).`ICD Code` AS ICD_Code,
ai_extract(table_html, array('Description')).`Description` AS Description,
ai_extract(table_html, array('Billed Amount')).`Billed Amount` AS Billed_Amount,
ai_extract(table_html, array('Paid Amount')).`Paid Amount` AS Paid_Amount
FROM structured_tables

In [ ]:
WITH rows AS (
SELECT *,
explode(regexp_extract_all(table_html, '<tr>(.*?)</tr>', 1)) as tr
FROM structured_tables
), data_rows AS (
SELECT path, tr
FROM rows
WHERE tr LIKE '%<td>%'
)
SELECT
ai_extract(tr, array('CPT Code')).`CPT Code` AS CPT_Code,
ai_extract(tr, array('ICD Code')).`ICD Code` AS ICD_Code,
ai_extract(tr, array('Description')).`Description` AS Description,
ai_extract(tr, array('Billed Amount')).`Billed Amount` AS Billed_Amount,
ai_extract(tr, array('Paid Amount')).`Paid Amount` AS Paid_Amount
FROM data_rows